In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv("/Users/ajaymundari/Downloads/Project/FoodHub/Data/Processed/orders_enriched.csv")

df.head()

,order_id,customer_id,restaurant_name,cuisine_type,cost_of_the_order,day_of_the_week,rating,food_preparation_time,delivery_time,signup_date,city,device_type,acquisition_channel,app_version,experiment_group
0,1477147,337525,Hangawi,Korean,30.75,Weekend,NaN,25,20,2023-12-19 07:40:22,Hyderabad,Android,Referral,4.8.1,Variant
1,1477685,358141,Blue Ribbon Sushi Izakaya,Japanese,12.08,Weekend,NaN,25,23,2023-12-25 16:13:39,Bangalore,Android,Organic,4.9.0,Control
2,1477070,66393,Cafe Habana,Mexican,12.23,Weekday,5.0,23,28,2023-09-13 08:16:28,Hyderabad,Android,Facebook,4.8.1,Control
3,1477334,106968,Blue Ribbon Fried Chicken,American,29.20,Weekend,3.0,25,15,2023-09-14 19:37:14,Bangalore,Android,Organic,4.9.0,Control
4,1478249,76942,Dirty Bird to Go,American,11.59,Weekday,4.0,25,24,2023-10-25 16:22:31,Hyderabad,Android,Organic,4.9.0,Variant


#### Step 4.2 – Generate Session IDs
Each order should belong to one app session.

In [2]:
df["session_id"] = [
                    f"S{100000+i}" for i in range(len(df))
                    ]

#### Step 4.3 – Generate Session Dates

In [3]:
start = pd.Timestamp("2024-01-01")
end = pd.Timestamp("2024-01-31")

df["session_date"] = pd.to_datetime(
    np.random.randint(
        start.value // 10**9,
        end.value // 10**9,
        len(df)
    ),
    unit="s"
)

#### Step 4.4 – Recommendation Seen
Assume almost everyone opening the app sees the recommendation widget.

In [4]:
df["recommendation_seen"] = np.random.choice(
                                            [1,0],
                                            size=len(df),
                                            p=[0.95,0.05]
                                            )

Step 4.5 – Recommendation Clicked This is where the A/B experiment begins. Control users click less often. Variant users click more often.

In [5]:
df["recommendation_clicked"] = np.where(
    df["experiment_group"]=="Control",
    np.random.binomial(1,0.20,len(df)),
    np.random.binomial(1,0.32,len(df))
)

#### Step 4.6 – Restaurants Viewed
Users who click recommendations generally browse more restaurants.

In [6]:
df["restaurants_viewed"] = np.where(
    df["recommendation_clicked"]==1,
    np.random.randint(4,9,len(df)),
    np.random.randint(1,5,len(df))
)

#### Step 4.7 – Cart Created
People who click recommendations are more likely to create a cart.

In [7]:
cart_probability = np.where(
                            df["recommendation_clicked"]==1,
                            0.80,
                            0.55
                            )

df["cart_created"] = np.random.binomial(
                                        1,
                                        cart_probability
                                        )

Step 4.8 – Order Completed


In [8]:
df["order_completed"] = 1

In [9]:
df["coupon_used"] = np.random.choice(
    [1,0],
    size=len(df),
    p=[0.35,0.65]
)

#### Validate the Experiment
Now let's verify that the Variant behaves better.
#### Click-Through Rate (CTR)

In [10]:
ctr = (
    df.groupby("experiment_group")["recommendation_clicked"]
      .mean()
      .mul(100)
      .round(2)
)

print(ctr)

experiment_group
Control    19.32
Variant    30.18
Name: recommendation_clicked, dtype: float64


#### Cart Creation Rate

In [11]:
cart = (
    df.groupby("experiment_group")["cart_created"]
      .mean()
      .mul(100)
      .round(2)
)

print(cart)

experiment_group
Control    60.82
Variant    64.04
Name: cart_created, dtype: float64


#### Save the Dataset

In [12]:
df.to_csv("/Users/ajaymundari/Downloads/Project/FoodHub/Data/Processed/product_events.csv",
    index=False
)